# EcoGuard
Sab cells upar se neeche ek-ek karke chalao. **Pehle config.py wale cell me PROJECT_ID badlo.**

In [ ]:
%%writefile config.py
"""EcoGuard settings - sab kuch yahin se badlo."""

PROJECT_ID = "your-gee-project-id"   # <-- apna Earth Engine / Google Cloud project ID likho

# Kin do saalon ka comparison (same mahine/season use karo)
YEAR_BEFORE = 2019
YEAR_AFTER = 2024

WATER_MONTHS = (10, 12)    # (start_month, end_month) - Oct-Dec = monsoon ke baad
FOREST_MONTHS = (10, 12)

CLOUD_MAX = 30             # % se zyada badal wali image skip

# Detection thresholds (result ajeeb lage to yahin tune karo)
WATER_THRESHOLD = 0.0      # MNDWI > 0  => paani
FOREST_NDVI = 0.5          # NDVI > 0.5 => ghana jungle/vegetation
SCALE_WATER = 20           # meter
SCALE_FOREST = 30          # meter

# Risk level (area me % badlaav): negative = ghata
RISK_HIGH = -25
RISK_MEDIUM = -10

# NOTE: coordinates approximate hain - Earth Engine map/Google Maps pe check kar lena
WATER_REGIONS = [
    {"name": "Sambhar Lake",     "lat": 26.97, "lon": 75.08, "buffer_km": 15},
    {"name": "Upper Lake Bhopal", "lat": 23.25, "lon": 77.35, "buffer_km": 6},
    {"name": "Loktak Lake",      "lat": 24.50, "lon": 93.80, "buffer_km": 12},
    {"name": "Kolleru Lake",     "lat": 16.65, "lon": 81.20, "buffer_km": 18},
]

FOREST_REGIONS = [
    {"name": "Hasdeo Arand",  "lat": 22.55, "lon": 82.50, "buffer_km": 20},
    {"name": "Aarey Colony",  "lat": 19.15, "lon": 72.88, "buffer_km": 5},
    {"name": "Similipal",     "lat": 21.85, "lon": 86.35, "buffer_km": 25},
]


In [ ]:
%%writefile gee_utils.py
"""Earth Engine ke chhote helper functions."""
import ee
import requests

import config as cfg

BANDS = ["B2", "B3", "B4", "B8", "B11", "SCL"]


def init_ee():
    """Earth Engine chalu karo. Pehli baar login maangega."""
    try:
        ee.Initialize(project=cfg.PROJECT_ID)
    except Exception:
        ee.Authenticate()
        ee.Initialize(project=cfg.PROJECT_ID)


def make_region(lat, lon, buffer_km):
    """Center point ke aas-paas ka square area."""
    return ee.Geometry.Point([lon, lat]).buffer(buffer_km * 1000).bounds()


def _mask_clouds(img):
    scl = img.select("SCL")
    bad = scl.eq(3).Or(scl.eq(8)).Or(scl.eq(9)).Or(scl.eq(10)).Or(scl.eq(11))
    return img.updateMask(bad.Not())


def get_composite(region, year, month_start, month_end):
    """Saal ke diye hue mahino ki cloud-free Sentinel-2 median image."""
    start = ee.Date.fromYMD(year, month_start, 1)
    end = ee.Date.fromYMD(year, month_end, 1).advance(1, "month")
    col = (
        ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED")
        .filterBounds(region)
        .filterDate(start, end)
        .filter(ee.Filter.lt("CLOUDY_PIXEL_PERCENTAGE", cfg.CLOUD_MAX))
        .select(BANDS)
    )
    n = col.size().getInfo()
    if n == 0:
        raise ValueError(f"{year} ({month_start}-{month_end}) me koi saaf image nahi mili")
    print(f"   {year}: {n} images")
    return col.map(_mask_clouds).median().clip(region)


def mndwi(img):
    return img.normalizedDifference(["B3", "B11"])


def ndvi(img):
    return img.normalizedDifference(["B8", "B4"])


def area_km2(mask, region, scale):
    """Mask (0/1 image) ka total area km2 me."""
    area = mask.rename("a").multiply(ee.Image.pixelArea())
    res = area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=region,
        scale=scale,
        maxPixels=1e10,
        bestEffort=True,
    )
    return ee.Number(res.get("a")).divide(1e6).getInfo()


def save_thumb(image, region, path, dims=800):
    """Image ko PNG me download karke save karo."""
    url = image.getThumbURL({"region": region, "dimensions": dims, "format": "png"})
    r = requests.get(url, timeout=180)
    r.raise_for_status()
    with open(path, "wb") as f:
        f.write(r.content)


def risk_level(change_pct):
    if change_pct <= cfg.RISK_HIGH:
        return "High"
    if change_pct <= cfg.RISK_MEDIUM:
        return "Medium"
    return "Low"


In [ ]:
%%writefile analysis.py
"""Ek region ka before/after change detection (water aur forest dono ke liye)."""
import ee

import config as cfg
from gee_utils import (make_region, get_composite, mndwi, ndvi,
                       area_km2, save_thumb, risk_level)

SPECS = {
    "water": {
        "label": "Water Body Shrinkage",
        "index": mndwi,
        "threshold": cfg.WATER_THRESHOLD,
        "months": cfg.WATER_MONTHS,
        "scale": cfg.SCALE_WATER,
        "colors": {"stable": "00B4D8", "lost": "FF0000", "gained": "FFD60A"},
    },
    "forest": {
        "label": "Deforestation",
        "index": ndvi,
        "threshold": cfg.FOREST_NDVI,
        "months": cfg.FOREST_MONTHS,
        "scale": cfg.SCALE_FOREST,
        "colors": {"stable": "1B5E20", "lost": "FF0000", "gained": "76FF03"},
    },
}


def slug(name):
    return name.lower().replace(" ", "_")


def analyze_region(module, reg, out_dir="results"):
    """module = 'water' ya 'forest'; reg = config wali dict. Ek row (dict) return karta hai."""
    spec = SPECS[module]
    region = make_region(reg["lat"], reg["lon"], reg["buffer_km"])

    c1 = get_composite(region, cfg.YEAR_BEFORE, *spec["months"])
    c2 = get_composite(region, cfg.YEAR_AFTER, *spec["months"])

    m1 = spec["index"](c1).gt(spec["threshold"]).unmask(0)
    m2 = spec["index"](c2).gt(spec["threshold"]).unmask(0)

    a1 = area_km2(m1, region, spec["scale"])
    a2 = area_km2(m2, region, spec["scale"])
    change = a2 - a1
    pct = (change / a1 * 100) if a1 > 0 else 0.0

    # Change map: stable / lost / gained
    col = spec["colors"]
    stable = m1.And(m2)
    lost = m1.And(m2.Not())
    gained = m1.Not().And(m2)

    def layer(mask, color):
        return mask.selfMask().visualize(palette=[color], min=0, max=1)

    base = c2.select(["B4", "B3", "B2"]).visualize(min=0, max=3000)
    final = ee.ImageCollection([
        base,
        layer(stable, col["stable"]),
        layer(lost, col["lost"]),
        layer(gained, col["gained"]),
    ]).mosaic()

    img_name = f"{module}_{slug(reg['name'])}.png"
    save_thumb(final, region, f"{out_dir}/images/{img_name}")

    return {
        "module": module,
        "region": reg["name"],
        "lat": reg["lat"],
        "lon": reg["lon"],
        "year_before": cfg.YEAR_BEFORE,
        "year_after": cfg.YEAR_AFTER,
        "area_before_km2": round(a1, 2),
        "area_after_km2": round(a2, 2),
        "change_km2": round(change, 2),
        "change_pct": round(pct, 2),
        "risk": risk_level(pct),
        "image": img_name,
    }


In [ ]:
%%writefile run_pipeline.py
"""Sab regions par analysis chalao aur results/ folder me save karo.

Colab me aise chalao (notebook cell me, !python se nahi):
    import ee; ee.Authenticate()
    from run_pipeline import main; main()
"""
import os

import pandas as pd

import config as cfg
from gee_utils import init_ee
from analysis import analyze_region

OUT = "results"


def main():
    init_ee()
    os.makedirs(f"{OUT}/images", exist_ok=True)

    jobs = [("water", r) for r in cfg.WATER_REGIONS] + \
           [("forest", r) for r in cfg.FOREST_REGIONS]

    rows = []
    for module, reg in jobs:
        print(f"[{module}] {reg['name']} ...")
        try:
            rows.append(analyze_region(module, reg, OUT))
            print("   done")
        except Exception as e:
            print(f"   FAILED: {e}")

    if not rows:
        print("Koi result nahi bana.")
        return None

    df = pd.DataFrame(rows)
    # priority: sabse zyada ghata hua region = rank 1 (module ke andar)
    df["priority_rank"] = df.groupby("module")["change_pct"].rank(method="min").astype(int)
    df = df.sort_values(["module", "priority_rank"])
    df.to_csv(f"{OUT}/summary.csv", index=False)
    print(f"\nSaved {OUT}/summary.csv")
    return df


if __name__ == "__main__":
    main()


In [ ]:
%%writefile app.py
"""EcoGuard - Streamlit dashboard (results/ folder se padhta hai, Earth Engine ki zaroorat nahi)."""
import os

import pandas as pd
import streamlit as st

CSV = "results/summary.csv"
IMG_DIR = "results/images"

MODULES = {"water": "💧 Water Body Shrinkage", "forest": "🌳 Deforestation"}
LEGEND = {
    "water": "🔵 Bacha hua paani  |  🔴 Paani khatam  |  🟡 Naya paani",
    "forest": "🟢 Bacha hua jungle  |  🔴 Jungle khatam  |  🟩 Naya hariyali",
}
RISK_ICON = {"High": "🔴", "Medium": "🟠", "Low": "🟢"}

st.set_page_config(page_title="EcoGuard", page_icon="🌍", layout="wide")
st.title("🌍 EcoGuard")
st.caption("Satellite (Sentinel-2) se paani aur jungle me badlaav ki nigrani")

if not os.path.exists(CSV):
    st.error("results/summary.csv nahi mili. Pehle run_pipeline.py chalao.")
    st.stop()

df = pd.read_csv(CSV)
module = st.sidebar.radio("Module", list(MODULES), format_func=MODULES.get)
d = df[df["module"] == module].sort_values("priority_rank").reset_index(drop=True)

if d.empty:
    st.warning("Is module ka data nahi hai.")
    st.stop()

years = f"{int(d.year_before[0])} → {int(d.year_after[0])}"
st.subheader(f"{MODULES[module]}  ({years})")

c1, c2, c3 = st.columns(3)
c1.metric("Regions", len(d))
c2.metric("High risk", int((d.risk == "High").sum()))
c3.metric("Sabse zyada badlaav", f"{d.change_pct.min():.1f}%", d.region[0])

left, right = st.columns([1, 1])
with left:
    st.markdown("**Map**")
    st.map(d.rename(columns={"lat": "latitude", "lon": "longitude"}), size=8000)
with right:
    st.markdown("**Badlaav (%) - region wise**")
    st.bar_chart(d.set_index("region")["change_pct"])

st.markdown("**Priority list** (rank 1 = sabse zyada nuksaan)")
table = d[["priority_rank", "region", "area_before_km2", "area_after_km2",
           "change_km2", "change_pct", "risk"]].copy()
table["risk"] = table["risk"].map(lambda r: f"{RISK_ICON.get(r, '')} {r}")
st.dataframe(table, use_container_width=True, hide_index=True)

st.divider()
choice = st.selectbox("Region ki detail dekho", d["region"])
row = d[d["region"] == choice].iloc[0]

m1, m2, m3 = st.columns(3)
m1.metric(f"Area {int(row.year_before)}", f"{row.area_before_km2} km²")
m2.metric(f"Area {int(row.year_after)}", f"{row.area_after_km2} km²",
          f"{row.change_km2} km²  ({row.change_pct}%)")
m3.metric("Risk", f"{RISK_ICON.get(row.risk, '')} {row.risk}")

img_path = os.path.join(IMG_DIR, row["image"])
if os.path.exists(img_path):
    st.image(img_path, caption=LEGEND[module], use_container_width=True)
else:
    st.info("Is region ki image nahi mili.")


In [ ]:
%%writefile requirements.txt
streamlit
pandas


## Earth Engine login

In [ ]:
import ee
ee.Authenticate()

## Sab regions par analysis (kuch minute lagenge)

In [ ]:
from run_pipeline import main
df = main()
df

## Results download karo

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')